# Case 2: Derinlemesine Veri Analizi Defteri

Case 1'de üretilen `src/services/analyzers/` modüllerini ve `merged_transactions.parquet`'i temel alıyor. İlk dört madde (numeric/categorical/datetime ayrımı, null ratio, outlier, distribution) Case 1'in analizleriyle büyük ölçüde örtüştüğü için **aynı fonksiyonlar yeniden kullanılıyor**; burada tekrar yazılmıyor, sadece Case 2'nin istediği çerçevede özetleniyor. Detaylı anlatı için `case_01_analysis.ipynb`'ye bakın. Kolon ilişkileri, rare categorical combination ve entity davranış pattern'leri (madde 5-7) tamamen yeni analizler, kendi bölümlerinde detaylı işleniyor.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise RuntimeError("repo root not found: expected a requirements.txt somewhere above " + str(start))


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/home/canberk/workspace/case_study')

In [2]:
import pandas as pd
import pyarrow.parquet as pq

from src.config import settings
from src.services.analyzers.column_types import profile_columns, classify_columns

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

parquet_path = settings.processed_data_path / "merged_transactions.parquet"
meta = pq.ParquetFile(parquet_path).metadata

profile = profile_columns(parquet_path)
classified = classify_columns(profile)

print(f"satır: {meta.num_rows}, kolon: {meta.num_columns}")

satır: 590540, kolon: 434


## 1. Numeric / Categorical / Datetime Ayrımı

`src/services/analyzers/type_grouping.py`, Case 1'in ince taneli `semantic_type`'ını (11 sınıf) üç üst kümeye topluyor; sınıflandırma mantığı tekrar yazılmıyor, sadece farklı bir açıdan gruplanıyor. Tek dikkat gerektiren karar: `numeric_encoded_categorical` (card1, addr1 gibi) sayısal dtype taşısa da **kategorik** kümeye giriyor; Case 1'de zaten "ortalaması alınmamalı" diye işaretlenmişti, burada da o karar korunuyor. `constant`/`near_constant` kolonlar `semantic_type`'tan sinyal almadığı için fiziksel dtype'a göre yerleştiriliyor.

In [3]:
from src.services.analyzers.type_grouping import group_columns_by_type

grouped = group_columns_by_type(classified)
grouped["high_level_type"].value_counts().to_frame(name="kolon sayısı")

,kolon sayısı
high_level_type,
categorical,307
numeric,124
excluded,2
datetime,1


In [4]:
grouped[grouped["high_level_type"] == "datetime"][["column", "semantic_type"]]

,column,semantic_type
2,TransactionDT,timestamp_offset


In [5]:
grouped[grouped["high_level_type"] == "excluded"][["column", "semantic_type"]]

,column,semantic_type
0,TransactionID,identifier
1,isFraud,target


Veri setinde tek bir gerçek zaman kolonu var (`TransactionDT`): bir timestamp değil, referans noktadan saniye cinsinden geçen süre (Case 1, bölüm 6'da detaylandırıldı). `TransactionID` (identifier) ve `isFraud` (target) bu üç kümenin dışında tutuluyor, ayrı ele alınıyorlar.

## 2. Null Ratio Analizi

Case 1'in `missingness.compute_missing_ratios()` fonksiyonu doğrudan yeniden kullanılıyor; Parquet'in row-group istatistiklerinden okunuyor, veri belleğe alınmadan. Eksiklik deseni gruplaması ve yapısal/rastgele ayrımının tam anlatısı `case_01_analysis.ipynb`, bölüm 1-3'te; burada sadece özet.

In [6]:
from src.services.analyzers.missingness import compute_missing_ratios

null_ratios = compute_missing_ratios(parquet_path)
null_ratios["bucket"].value_counts().to_frame(name="kolon sayısı")

,kolon sayısı
bucket,
very_high,208
low,92
moderate,90
high,24
none,20


## 3. Outlier Analizi

Case 1'in `quality.compute_outlier_ratios()` fonksiyonu, aynı sayısal kolon kümesiyle (`numeric_continuous` + `monetary` + `count`, 96 kolon) yeniden çalıştırılıyor: IQR (Tukey) ve MAD (modified z-score) yan yana. Detaylı yorum (id_25 tipleme-sınırı bulgusu dahil) Case 1, bölüm 5'te.

In [7]:
from src.services.analyzers.quality import compute_outlier_ratios

numeric_cols = classified[classified["semantic_type"].isin(["numeric_continuous", "monetary", "count"])]["column"].tolist()
outliers = compute_outlier_ratios(parquet_path, numeric_cols)
outliers.head(10)

,column,iqr_outlier_ratio,mad_outlier_ratio
0,id_25,0.312549,0.377046
1,V263,0.246454,NaN
2,C8,0.241936,NaN
3,V202,0.239703,NaN
4,V166,0.238093,NaN
5,V165,0.236153,NaN
6,V333,0.235650,NaN
7,C4,0.234797,NaN
8,C10,0.232157,NaN
9,V128,0.230442,NaN


## 4. Distribution Analizi

Case 1'in `distributions.py` fonksiyonları yeniden kullanılıyor: sayısal (percentile, çarpıklık, log-dönüşüm önerisi) ve kategorik (entropi, dengesizlik oranı) taraf ayrı. Zaman ekseni (saat bazlı fraud oranı grafiği) ve tam anlatı Case 1, bölüm 6'da.

In [8]:
from src.services.analyzers.distributions import (
    compute_numeric_distributions,
    compute_categorical_distributions,
    NUMERIC_SEMANTIC_TYPES,
    CATEGORICAL_SEMANTIC_TYPES,
)

numeric_dist = compute_numeric_distributions(parquet_path, classified[classified["semantic_type"].isin(NUMERIC_SEMANTIC_TYPES)]["column"].tolist())
categorical_dist = compute_categorical_distributions(parquet_path, classified[classified["semantic_type"].isin(CATEGORICAL_SEMANTIC_TYPES)]["column"].tolist())

print(f"sayısal: {len(numeric_dist)} kolon, kategorik: {len(categorical_dist)} kolon")
numeric_dist.head(5)[["column", "skewness", "log_transform_recommended"]]

sayısal: 96 kolon, kategorik: 305 kolon


,column,skewness,log_transform_recommended
0,V311,323.831374,True
1,V129,240.274253,True
2,V309,224.875268,True
3,V206,207.881617,True
4,V319,181.833654,True


---

**Durum:** madde 1-4 tamamlandı (Case 1'den yeniden kullanılarak). Madde 5-7 (rare categorical combination, kolon ilişkileri, entity davranış pattern'leri) yeni analizler; sıradaki bölümlerde ekleniyor.

## 5. Rare Categorical Combination Analizi

Tek kolonun içindeki nadir değerler değil, **kolonlar arası birlikte görülme** nadirliği araştırılıyor; her değer tek başına yaygın olsa da belirli bir kombinasyon çok az görülebilir; bunu tek-kolon profilleri (Case 1) göremez.

`src/services/analyzers/rare_combinations.py`, kasıtlı olarak küçük, yorumlanabilir bir kolon kümesiyle çalışıyor (`ProductCD`, `card4`, `card6`, isteğe bağlı `DeviceType`); 307 kategorik kolonun tamamını kombine etmek hem hesaplama açısından anlamsız hem yorumlanamaz olurdu (kombinasyon uzayı çarpımsal büyüyor). Bir kombinasyon **nadir** sayılıyor eğer: satır sayısı ≤10 **veya** oranı ≤%0,01 (`RARE_COMBINATION_MAX_COUNT`, `RARE_COMBINATION_MAX_RATIO`).

In [9]:
from src.services.analyzers.rare_combinations import compute_combination_frequencies, find_unobserved_combinations

freq_3col = compute_combination_frequencies(parquet_path, columns=["ProductCD", "card4", "card6"])
print(f"gözlenen kombinasyon: {len(freq_3col)}")
freq_3col.head(10)

gözlenen kombinasyon: 37


,ProductCD,card4,card6,count,ratio,is_rare
0,C,american express,credit,2,0.000003,True
1,R,american express,charge card,3,0.000005,True
2,H,discover,debit,7,0.000012,True
3,R,discover,debit,11,0.000019,True
4,C,visa,charge card,12,0.000020,True
5,S,american express,debit,27,0.000046,True
6,W,mastercard,debit or credit,30,0.000051,True
7,H,american express,debit,48,0.000081,True
8,R,american express,debit,69,0.000117,False
9,W,discover,debit,329,0.000557,False


### Önce üç kolonla (identity gerektirmeyen)

`ProductCD`, `card4`, `card6`: üçü de transaction tablosundan, identity eşleşmesi gerektirmiyor. Nadir kombinasyonlar arasında ilginç bir tanesi: `card6="debit or credit"` (card6'nın 4. ve en nadir değeri) neredeyse tamamen `ProductCD="W"` ile birlikte görülüyor.

In [10]:
freq_3col[freq_3col["is_rare"]].reset_index(drop=True)

,ProductCD,card4,card6,count,ratio,is_rare
0,C,american express,credit,2,0.000003,True
1,R,american express,charge card,3,0.000005,True
2,H,discover,debit,7,0.000012,True
3,R,discover,debit,11,0.000019,True
4,C,visa,charge card,12,0.000020,True
5,S,american express,debit,27,0.000046,True
6,W,mastercard,debit or credit,30,0.000051,True
7,H,american express,debit,48,0.000081,True


### Dördüncü kolon eklenince: `DeviceType`, beklenmedik bir bulgu

`DeviceType` identity tablosundan geliyor ve sadece identity eşleşmesi olan satırlarda dolu (\~%24,4). Bunu kombinasyona eklediğimizde:

In [11]:
freq_4col = compute_combination_frequencies(parquet_path, columns=["ProductCD", "card4", "card6", "DeviceType"])
freq_4col["ProductCD"].value_counts()

ProductCD
R    18
H    16
S    14
C    10
Name: count, dtype: int64

**`ProductCD="W"` dördüncü kolonlu tabloda hiç görünmüyor.** İlk bakışta bir hata gibi duruyor ama gerçek: 

In [12]:
df_check = pq.ParquetFile(parquet_path).read(columns=["ProductCD", "DeviceType"]).to_pandas()

print("ProductCD dağılımı, tüm veri:")
print(df_check["ProductCD"].value_counts())
print()
print("ProductCD dağılımı, sadece DeviceType dolu olan (identity eşleşmiş) satırlarda:")
print(df_check[df_check["DeviceType"].notna()]["ProductCD"].value_counts())

ProductCD dağılımı, tüm veri:
ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64

ProductCD dağılımı, sadece DeviceType dolu olan (identity eşleşmiş) satırlarda:
ProductCD
C    61015
R    36429
H    32098
S    11268
Name: count, dtype: int64


**Bulgu:** `ProductCD="W"` olan 439.670 işlemin (veri setinin **%74,5'i**) **hiçbirinde** identity eşleşmesi yok; `DeviceType` dahil tüm identity kolonları bu satırlarda tamamen boş. Bu, Case 1 bölüm 1'deki genel %24,4 identity kapsama oranının ürün koduna göre son derece eşitsiz dağıldığını gösteriyor: `C`/`R`/`H`/`S` ürünlerinde identity kapsaması ciddi (61015/68519 ≈ %89, `C` için), `W` için ise pratikte **%0**.

Bu, dört kolonlu kombinasyon tablosunun (`DeviceType` dahil) **veri setinin çoğunluğunu (W ürünü) örtük biçimde dışladığı** anlamına geliyor; metodolojik olarak açıkça belirtilmesi gereken bir sınırlama. Bu yüzden nadir kombinasyon sonucu iki ayrı görünüm olarak sunuluyor: (a) tüm veriyi kapsayan 3 kolonlu görünüm, (b) sadece identity-eşleşmiş alt kümeyi kapsayan 4 kolonlu görünüm; ikisi karıştırılmamalı.

In [13]:
freq_4col[freq_4col["is_rare"]].reset_index(drop=True)

,ProductCD,card4,card6,DeviceType,count,ratio,is_rare
0,R,american express,charge card,desktop,1,0.000002,True
1,H,discover,debit,desktop,2,0.000003,True
2,R,american express,charge card,mobile,2,0.000003,True
3,S,american express,debit,mobile,2,0.000003,True
4,R,discover,debit,desktop,3,0.000005,True
5,C,visa,charge card,mobile,4,0.000007,True
6,H,discover,debit,mobile,5,0.000008,True
7,C,visa,charge card,desktop,8,0.000014,True
8,R,discover,debit,mobile,8,0.000014,True
9,H,american express,debit,desktop,23,0.000039,True


### Hiç gözlenmemiş kombinasyonlar (3 kolonlu, tüm veri)

In [14]:
unobserved = find_unobserved_combinations(parquet_path, columns=["ProductCD", "card4", "card6"])
print(f"{len(unobserved)} kombinasyon hiç gözlenmedi")
unobserved

43 kombinasyon hiç gözlenmedi


,ProductCD,card4,card6
0,C,american express,charge card
1,C,american express,debit
2,C,american express,debit or credit
3,C,discover,charge card
4,C,discover,credit
5,C,discover,debit
6,C,discover,debit or credit
7,C,mastercard,charge card
8,C,mastercard,debit or credit
9,C,visa,debit or credit


---

**Durum:** madde 5 tamamlandı. Beklenmedik ama önemli bir yan bulgu: identity kapsaması `ProductCD`'ye göre son derece eşitsiz (`W` için \~%0, diğerleri için çok daha yüksek); bu, madde 6 (kolon ilişkileri) ve madde 7'ye (entity davranış pattern'leri) taşınacak bir sinyal.

## 6. Kolon İlişkilerini Keşfediniz

`src/services/analyzers/relationships.py`, üç farklı ilişki türünü ayrı matematikle ele alıyor:

1. **Sayısal-sayısal**: Pearson (doğrusal) + Spearman (monotonik, sıra tabanlı) korelasyon, 96 sayısal kolonun tamamı arasında. İkisi birlikte hesaplanıyor çünkü Case 1'in bulduğu aşırı çarpık dağılımlarda (bir avuç uç değerin domine ettiği V-kolonları) ikisi çok farklı sonuç verebiliyor.
2. **Kategorik-kategorik**: Cramér's V (contingency-table ilişki gücü, scipy'siz; chi-square doğrudan gözlenen/beklenen sayılardan hesaplanıyor). `rare_combinations.py`'deki aynı gerekçeyle, sabit ve yorumlanabilir 15 kolonluk bir kümeyle sınırlı (`ProductCD`, `card4`, `card6`, `DeviceType`, e-posta alanları, `M1..M9`); 307 kategorik kolonun tamamını taramak hem yavaş olurdu hem çoğunlukla V-ailesi içi beklenen ilişkileri gösterirdi, gerçek çapraz-domain ilişkileri boğardı.
3. **Sayısal-kategorik**: eta-squared (bir sayısal kolonun varyansının ne kadarının kategorik gruplar arasında olduğu), 96 sayısal kolon × aynı 15 kategorik kolon.

Sayısal-sayısal korelasyon tek istisna: bu, kolon batch'leriyle değil **tüm 96 kolonu aynı anda** okuyarak hesaplanıyor (korelasyon çift-kolonlu bir istatistik, batch'lenemez); bellek riskini azaltmak için float64 yerine float32 olarak okunuyor (\~560MB yerine \~280MB).

In [15]:
from src.services.analyzers.relationships import (
    compute_numeric_correlations,
    compute_categorical_associations,
    compute_numeric_categorical_relationships,
    RELATIONSHIP_CATEGORICAL_COLUMNS,
)

numeric_cols_96 = classified[classified["semantic_type"].isin(["numeric_continuous", "monetary", "count"])]["column"].tolist()
correlations = compute_numeric_correlations(parquet_path, numeric_cols_96)
print(f"{len(correlations)} çift, |Pearson| veya |Spearman| >= 0.7")
correlations.head(15)

664 çift, |Pearson| veya |Spearman| >= 0.7


,column_a,column_b,pearson,spearman
0,C7,C12,0.999489,0.804057
1,V132,V316,0.997222,0.718718
2,C8,C10,0.996970,0.970461
3,V127,V332,0.996655,0.911223
4,V307,V332,0.996515,0.856394
5,C1,C11,0.996515,0.737083
6,V128,V333,0.996275,0.917991
7,V266,V269,0.996097,0.467048
8,V133,V332,0.996004,0.562091
9,V317,V332,0.995765,0.517847


Çoğu beklenen: `C` kolonları (sayaç özellikleri) birbiriyle çok yüksek korele: `C7`\~`C12` (Pearson 0,999), `C1`\~`C11` (0,997), muhtemelen aynı temel işlem sayısının farklı türevleri. V-kolon çiftleri de benzer şekilde bir "aile" içinde yüksek korele.

**Dikkat çeken satır: `V266`\~`V269`.** Pearson **0,996** ama Spearman sadece **0,467**; aradaki fark 0,53, tablodaki en büyük ayrışma. Bu, iki yöntemi neden birlikte kullandığımızın somut kanıtı:

In [16]:
divergence = correlations.copy()
divergence["divergence"] = (divergence["pearson"] - divergence["spearman"]).abs()
divergence.sort_values("divergence", ascending=False).head(10)

,column_a,column_b,pearson,spearman,divergence
299,C4,C14,0.907676,-0.249540,1.157216
102,C1,C8,0.967746,-0.165596,1.133342
101,C1,C4,0.967800,-0.156873,1.124674
111,C4,C6,0.962319,-0.159304,1.121624
382,C8,C14,0.860246,-0.256649,1.116895
120,C1,C10,0.958202,-0.154784,1.112986
275,C6,C8,0.921972,-0.178712,1.100683
393,C10,C14,0.853009,-0.245464,1.098473
288,C6,C10,0.914440,-0.172309,1.086749
108,C8,C11,0.962722,-0.083727,1.046448


Pearson doğrusal ilişkiyi ölçüyor ve bir avuç aşırı uç değer tarafından domine edilebiliyor (Case 1, bölüm 5-6'da bu V-kolonlarının %92-98 sıfır, seyrek çok büyük değerli olduğunu görmüştük); birkaç eş-büyük uç değer Pearson'ı yapay olarak şişirebilir. Spearman sıraya dayalı olduğu için bu tür uç değerlere karşı sağlam; 0,467'lik değer, satırların büyük çoğunluğunda (çoğunlukla sıfır oldukları için) gerçek bir sıralama ilişkisi kadar güçlü olmadığını gösteriyor. Sonuç: `V266`\~`V269` arasındaki "ilişki", büyük ölçüde birkaç ortak uç değerden kaynaklanıyor, genel bir monotonik eğilimden değil.

### Kategorik-kategorik ilişkiler (Cramér's V)

In [17]:
associations = compute_categorical_associations(parquet_path)
associations[associations["is_strong"]].reset_index(drop=True)

,column_a,column_b,cramers_v,is_strong
0,ProductCD,M4,0.857534,True
1,P_emaildomain,R_emaildomain,0.697980,True
2,M2,M3,0.665161,True
3,M7,M8,0.493290,True
4,DeviceType,M4,0.420369,True
5,M2,M9,0.408110,True
6,ProductCD,R_emaildomain,0.355895,True
7,P_emaildomain,M4,0.339180,True
8,M8,M9,0.323376,True
9,ProductCD,P_emaildomain,0.308454,True


En güçlü ilişki `ProductCD`\~`M4` (Cramér's V = 0,86): `M4` alanı büyük ölçüde `ProductCD`'ye bağımlı görünüyor (muhtemelen bazı ürün kodlarında bu alan hiç doldurulmuyor ya da sabit bir değer alıyor). `P_emaildomain`\~`R_emaildomain` (0,70) da mantıklı: gönderen ve alıcı genelde benzer e-posta sağlayıcılarını kullanıyor. `M2`\~`M3` (0,67) ve `M7`\~`M8` (0,49) M-bayrakları arası kendi içinde kümelenme gösteriyor.

### Sayısal-kategorik ilişkiler (eta-squared)

In [18]:
numeric_categorical = compute_numeric_categorical_relationships(parquet_path, numeric_cols_96)
numeric_categorical[numeric_categorical["is_strong"]].head(15).reset_index(drop=True)

,numeric_column,categorical_column,eta_squared,is_strong
0,id_02,ProductCD,0.237087,True
1,V160,ProductCD,0.164624,True
2,V159,ProductCD,0.155554,True
3,id_21,DeviceType,0.142453,True
4,V128,ProductCD,0.093655,True
5,TransactionAmt,R_emaildomain,0.092077,True
6,V308,ProductCD,0.091079,True
7,V335,ProductCD,0.090647,True
8,V134,ProductCD,0.089483,True
9,V127,ProductCD,0.088359,True


`TransactionAmt`\~`R_emaildomain` (eta² = 0,092) iş açısından en yorumlanabilir olanı: işlem tutarı, alıcının e-posta sağlayıcısına göre anlamlı ölçüde değişiyor. `id_02`\~`ProductCD` (0,237) en güçlüsü ama `id_02` identity tablosundan geldiği için bu ilişki de dolaylı olarak identity kapsamasının `ProductCD`'ye göre eşitsiz dağılımıyla (bölüm 5'teki `W` bulgusu) bağlantılı olabilir; kesin nedensellik iddia edilmiyor, sadece istatistiksel ilişki raporlanıyor.

---

**Durum:** madde 6 tamamlandı.

## 7. Entity Davranış Pattern'lerini Analiz Ediniz

Case 1'in cardinality analizinde "varlık anahtarı olarak koru" diye işaretlenen 10 kolonu şimdi gerçekten kullanıyoruz. Birincil entity olarak `card1` seçildi: 10 varlık anahtarı arasında en yüksek kardinaliteli (13.553 farklı değer) ve en çok "müşteri kimliği"ne yakın olanı. `addr1` ve `DeviceInfo`, ayrı entity olarak değil, `card1`'in davranış boyutları olarak (kaç farklı bölge/cihazdan işlem yaptığı) dahil ediliyor; tek başına bir bölge kodunun "davranışı" olmaz, tekrarlayan bir kartın olur.

`src/services/analyzers/entity_behavior.py`, `groupby().agg()` + `groupby().diff()` ile tamamen vektörel çalışıyor (13.553 grup için satır satır Python fonksiyonu çağırmak yavaş olurdu). Tek işlemi olan entity'ler (`transaction_count < 2`) profillemeden çıkarılıyor; tek veri noktasının "pattern"i olmaz. `isFraud` yine sadece betimleyici (entity başına fraud oranı); hangi entity'nin işaretleneceğine dair hiçbir eşik etikete bakmıyor.

In [19]:
from src.services.analyzers.entity_behavior import (
    build_entity_profiles,
    flag_unusual_entities,
    summarize_entity_profiles,
)

profiles = build_entity_profiles(parquet_path)
summarize_entity_profiles(profiles)

{'profiled_entities': 10109,
 'median_transactions_per_entity': 7.0,
 'max_transactions_per_entity': 14932,
 'entities_with_multiple_addr1': 4601,
 'entities_with_multiple_devices': 3792}

590.540 işlemin 10.109'u (birden fazla işlemi olan `card1` değerleri) profillenebildi. İşlem sayısı dağılımına bakalım:

In [20]:
profiles["transaction_count"].describe()

count    10109.000000
mean        58.076565
std        379.957238
min          2.000000
25%          3.000000
50%          7.000000
75%         21.000000
max      14932.000000
Name: transaction_count, dtype: float64

### Beklenmedik bulgu: `card1` yüksek hacimde temiz bir kart kimliği gibi davranmıyor

Medyan 7 işlem ama **maksimum 14.932**; p99 sadece \~1014. Yani entity'lerin %1'i 1014'ten fazla işleme sahip, ama en tepedeki tek bir değer bunun \~15 katı. Bu ölçekte bir tekil "kart"ın davranışı olamayacağını düşündürüyor; kontrol edelim:

In [21]:
profiles.nlargest(15, "transaction_count")[
    ["entity", "transaction_count", "distinct_addr1", "distinct_device_info", "fraud_rate_pct"]
]

,entity,transaction_count,distinct_addr1,distinct_device_info,fraud_rate_pct
3964,7919,14932,57,9,0.750067
4904,9500,14162,54,67,3.728287
8632,15885,10361,32,614,4.285301
9415,17188,10344,54,70,2.687548
8152,15066,7945,62,90,3.939585
6793,12695,7091,38,42,2.834579
6688,12544,6773,37,42,2.155618
2865,6019,6771,64,129,4.342047
1039,2803,6141,63,59,1.188731
3777,7585,5334,56,78,4.930634


**Bulgu:** en yüksek hacimli `card1` değerleri onlarca farklı bölgeden (`distinct_addr1` 28-64 arası) ve, daha çarpıcısı, **yüzlerce farklı cihazdan** (`card1=15885`: 614 farklı `DeviceInfo`; `card1=3154`: 412; `card1=9633`: 398) işlem gösteriyor. Tek bir fiziksel kart 614 farklı cihazdan kullanılmaz.

**Yorum (dürüstçe, kesin nedensellik iddia etmeden):** `card1`, düşük-orta hacimli değerlerde muhtemelen gerçek bir kart kimliğine yakın davranıyor (medyan 7 işlem, makul), ama en yüksek hacimli değerler istatistiksel olarak bir "bucket" (birden fazla gerçek kartın aynı anonimleştirilmiş değere düştüğü bir toplu kutu) gibi görünüyor; Vesta'nın anonimleştirme şeması card1'i sınırlı bir değer aralığına (bölüm 4'te görülen 1000-18396) sıkıştırdığı için bu beklenmedik değil. **Pratik sonuç:** sonraki anomali tespiti katmanında `card1` bazlı velocity/çeşitlilik özellikleri üretilirken, yüksek hacimli `card1` değerleri için bu özellikler "bir entity'nin gerçek davranışı" değil "bir bucket'ın toplu gürültüsü" olarak yorumlanmalı; düşük-orta hacimli entity'lerde bu risk çok daha azdır.

### Davranış bayrakları

Üç eşik: `distinct_addr1 >= 3`, `amount_cv >= 1.5` (harcama tutarında yüksek değişkenlik), `min_gap_seconds <= 60` (art arda işlemler arasında bir dakikadan kısa boşluk: hız sinyali).

In [22]:
flagged = flag_unusual_entities(profiles)
flagged["unusual_flag_count"].value_counts().sort_index().to_frame(name="entity sayısı")

,entity sayısı
unusual_flag_count,
0,6296
1,2305
2,1226
3,282


In [23]:
flagged[flagged["unusual_flag_count"] == 3].sort_values("transaction_count", ascending=False).head(10).reset_index(drop=True)

,entity,transaction_count,distinct_addr1,distinct_device_info,amount_mean,amount_std,min_gap_seconds,median_gap_seconds,fraud_rate_pct,amount_cv,high_addr_diversity,high_amount_variability,fast_velocity,unusual_flag_count
0,9500,14162,54,67,115.698119,193.372544,0.0,504.0,3.728287,1.671354,True,True,True,3
1,12695,7091,38,42,141.144645,215.282797,0.0,1027.0,2.834579,1.525264,True,True,True,3
2,6019,6771,64,129,225.441337,395.988543,0.0,713.5,4.342047,1.756504,True,True,True,3
3,2803,6141,63,59,142.683409,261.983056,0.0,1182.0,1.188731,1.836114,True,True,True,3
4,7585,5334,56,78,206.714779,322.611827,0.0,1122.0,4.930634,1.560662,True,True,True,3
5,10616,5172,56,69,256.983517,388.538923,0.0,1165.0,3.905646,1.511922,True,True,True,3
6,12839,5129,47,35,120.725436,193.922468,1.0,1391.5,1.364788,1.606310,True,True,True,3
7,18132,4209,34,34,123.416340,192.717425,1.0,1746.5,1.401758,1.561523,True,True,True,3
8,16132,3929,40,41,162.273696,250.472147,0.0,1760.5,1.628913,1.543517,True,True,True,3
9,16075,3748,56,57,279.848370,857.792210,0.0,1564.0,4.855923,3.065204,True,True,True,3


282 entity (profillenen 10.109'un \~%2,8'i) üç bayrağı da taşıyor. Beklendiği gibi bunlar yukarıdaki en yüksek hacimli entity'lerle büyük ölçüde örtüşüyor; bucket yorumuyla tutarlı: yüksek hacim, doğal olarak daha fazla bölge/cihaz/hız çeşitliliği demek.

---

**Durum:** madde 7 tamamlandı; Case 2'nin yedi maddesi de bitti.

## Sonuç: Case 2 özeti

| # | Analiz | Kaynak | Ana bulgu |
|---|---|---|---|
| 1 | Numeric/categorical/datetime ayrımı | `type_grouping.py` (yeni) | 307 kategorik, 124 sayısal, 1 datetime, 2 hariç |
| 2 | Null ratio | Case 1'den yeniden kullanım | (değişmedi) |
| 3 | Outlier | Case 1'den yeniden kullanım | (değişmedi) |
| 4 | Distribution | Case 1'den yeniden kullanım | (değişmedi) |
| 5 | Rare categorical combination | `rare_combinations.py` (yeni) | `ProductCD="W"` (%74,5) identity/DeviceType eşleşmesinden tamamen yoksun; beklenmedik yan bulgu |
| 6 | Kolon ilişkileri | `relationships.py` (yeni) | `V266`\~`V269`: Pearson 0,996 vs Spearman 0,467 (uç değer kaynaklı sahte doğrusal ilişki); `ProductCD`\~`M4` en güçlü kategorik ilişki (0,86) |
| 7 | Entity davranış pattern'leri | `entity_behavior.py` (yeni) | `card1` düşük-orta hacimde kart kimliğine yakın, yüksek hacimde muhtemelen anonimleştirme bucket'ı (614 farklı cihaz gözlemlenen tek değer) |

**Case 1 → Case 2 zinciri:** Case 1'in ürettiği `classified` (semantik tip) ve "varlık anahtarı" listesi, Case 2'nin hem kapsam kararlarını (hangi kolonlarla ilişki/kombinasyon analizi yapılacağı) hem asıl analiz nesnesini (entity davranışı) doğrudan besledi; iki case ayrı çalışmadı, ikincisi birincinin üzerine inşa edildi.